# DATA 37100 — Final Project Analysis Template

Use this notebook to produce your analysis deliverable.

**Expected effort:** ~20 hours per student

## What to include
- Baseline summaries for **two** model families (GAN/DCGAN, Transformer, Diffusion)
- One **two‑knob** controlled experiment (exactly two factors)
- Side-by-side sample visualization (grids or text snippets)
- Evidence-based interpretation + at least one failure mode


## 1. Setup
Run from the **repo root**. Adjust paths if needed.

In [ ]:
from pathlib import Path
import json
import pandas as pd

REPO_ROOT = Path('.').resolve()

# Diffusion outputs (default)
DIFF_OUT = REPO_ROOT / 'untrack/outputs/final/diffusion'

# Transformer outputs (default)
XFORM_OUT = REPO_ROOT / 'untrack/outputs/final/transformer'

DIFF_OUT, XFORM_OUT

## 2. Select your experiment manifest (results.csv)

If you ran a grid experiment, you should have a `results.csv` with a `run_dir` column.

In [ ]:
# Choose ONE
results_csv = DIFF_OUT / 'results.csv'   # or XFORM_OUT / 'results.csv'
print(results_csv)
df = pd.read_csv(results_csv)
df.head()

## 3. Quick run summary table
Each run directory should contain `run_args.json` and `summary.json`.

In [ ]:
def read_json(p: Path):
    return json.loads(p.read_text(encoding='utf-8'))

rows = []
for rd in df['run_dir']:
    rd = Path(rd)
    args = read_json(rd / 'run_args.json') if (rd / 'run_args.json').exists() else {}
    summ = read_json(rd / 'summary.json') if (rd / 'summary.json').exists() else {}
    row = {'run_dir': str(rd), 'name': rd.name}
    row.update({k: args.get(k) for k in ['dataset','T','target','beta2','base_ch','time_emb_dim','temperature','top_p','d_model','n_heads','n_layers'] if k in args})
    row.update({'seconds': summ.get('seconds'), 'device': summ.get('device')})
    rows.append(row)

summary_df = pd.DataFrame(rows)
summary_df

## 4. Visualize samples

For diffusion runs, the starter saves PNG grids in each run directory (or `samples/`).
We'll load the first grid we can find for each run and show a contact sheet.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

def find_pngs(run_dir: Path):
    candidates = []
    if (run_dir / 'samples').exists():
        candidates += sorted((run_dir / 'samples').glob('*.png'))
    candidates += sorted(run_dir.glob('*.png'))
    candidates = sorted(candidates, key=lambda p: (('grid' not in p.name.lower() and 'sample' not in p.name.lower()), p.stat().st_mtime))
    return candidates

imgs = []
titles = []
for rd in df['run_dir']:
    rd = Path(rd)
    pngs = find_pngs(rd)
    if pngs:
        imgs.append(pngs[0])
        titles.append(rd.name)

n = len(imgs)
ncols = 3
nrows = (n + ncols - 1) // ncols
plt.figure(figsize=(4*ncols, 4*nrows))
for i, (p, t) in enumerate(zip(imgs, titles), start=1):
    ax = plt.subplot(nrows, ncols, i)
    ax.imshow(mpimg.imread(str(p)))
    ax.set_title(t, fontsize=9)
    ax.axis('off')
plt.tight_layout()

## 5. Simple diversity proxy (optional)

A lightweight metric for diffusion samples: average pairwise L2 distance in pixel space on a subset.

*This is intentionally simple and fast.*

In [ ]:
import numpy as np

def diversity_proxy(x: np.ndarray, max_items: int = 64) -> float:
    # x: (N,H,W) or (N,H,W,C)
    x = x[:max_items].reshape(min(len(x), max_items), -1).astype(np.float32)
    n = x.shape[0]
    if n < 2:
        return float('nan')
    # sample a few pairs
    idx = np.random.default_rng(0).integers(0, n, size=(min(512, n*n), 2))
    d = np.linalg.norm(x[idx[:,0]] - x[idx[:,1]], axis=1)
    return float(d.mean())

print('Define how to load raw samples here if your run saves tensors; otherwise skip this section.')


## 6. Failure modes & limitations (required)

Write short, evidence-based notes:
- What artifacts appeared? When?
- Which knob changes increased/decreased failures?
- What does this imply about model assumptions?


## 7. Conclusions (required)

In 5–8 bullet points:
- Key findings
- One surprising result
- One limitation
- One next step
